In [0]:
%run "../01-setup/1.configure_access_to_cloud_storage"

In [0]:
race_results_df = spark.read.format("delta").load(f"{presentation_folder_path}/race_results")
race_results_df.printSchema()

In [0]:
from pyspark.sql.functions import sum, when, count, col, desc
driver_standings_df = race_results_df \
  .groupBy("race_year", "driver_name", "driver_nationality", "team") \
  .agg(sum("points").alias("total_points"), 
       count(when(col("position") == 1, True)).alias("wins")) \
  .orderBy(desc("total_points"))

display(driver_standings_df)
  

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import desc, rank

driver_rank_spec = Window.partitionBy("race_year").orderBy(desc("total_points"), desc("wins"))
final_df = driver_standings_df.withColumn("rank", rank().over(driver_rank_spec))
display(final_df)

In [0]:
final_df.write \
  .mode("overwrite") \
  .partitionBy("race_year") \
  .format("delta") \
  .save(f"{presentation_folder_path}/driver_standings")